# Lesson 02 - Exploring Microsoft Agent Framework

Di **Microsoft Agent Framework (MAF)** na one framework wey dey united for building AI agents. E get clean, composable architecture wey get four core building blocks:

- **Client** – e dey connect to AI model endpoint and e dey handle communication
- **Agent** – e dey wrap client with instructions and tool definitions
- **Tools** – dem dey extend agent capabilities with custom functions wey di model fit call
- **Session** – e dey maintain conversation history for multi-turn interactions

For dis lesson, we go build **travel booking agent** wey go check destination availability using these concepts.


## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Understanding the Agent Framework Architecture

Di Microsoft Agent Framework dey follow one kind layered architecture:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – One `AzureAIProjectAgentProvider` dey connect to Azure OpenAI deployment. E dey handle authentication, request formatting, and response parsing.
2. **Agent** – E dey created from di client through `provider.create_agent()`, di agent dey combine model access with instructions (system prompt) and tools dem.
3. **Tools** – Na Python functions wey dem put `@tool` decorate for, wey di agent fit call come perform actions or to find data.
4. **Session** – One `AgentSession` object (wey dem create through `agent.create_session()`) wey dey store conversation history, e dey make multi-turn dialogue possible where di agent dey remember wetin don happen before.

Make we build each layer step by step.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adding Tools with the @tool Decorator

Tools dey allow agents to take actions wey pass just to dey generate text. The `@tool` decorator dey change normal Python function to something wey agent fit call.

Key points:
- Use `Annotated[type, "description"]` so the model fit understand every parameter.
- The docstring go become the tool description wey the model go see.
- `approval_mode="never_require"` mean say the tool go run automatically without need user confirmation.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creating an Agent with Tools

Now we combine di client, instructions, and tools into one agent. Di `instructions` dey act as di system prompt — dem dey define di agent pesona and how e go dey behave.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Multi-Turn Conversations wit Sessions

An `AgentSession` (wey dem create using `agent.create_session()`) dey keep track of all messages wey dey for one conversation. If you pass the same session to every `agent.run()` call, the agent go get access to all the conversation history and fit refer back to messages wey dem talk before.

We dey pass `tools=[check_destination_availability]` make the agent fit call our availability checker every time e turn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

For dis lesson you explore di four pillars of di Microsoft Agent Framework:

| Concept | Wetin You Learn |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` dey connect to Azure OpenAI wit credential-based auth |
| **Agent** | `provider.create_agent()` bundle model connection wit instructions and name |
| **Tools** | Di `@tool` decorator dey show Python functions wey agent fit call |
| **Session** | `agent.create_session()` dey maintain conversation history across multiple turns |

Dem building blocks combine make agents wey fit hold natural conversations, call external functions, and maintain context — di foundation for more advanced agentic patterns for later lessons.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even though we try make e correct, make you sabi say automated translation fit get mistake or no too accurate. Di original document wey e original language na di correct one wey you suppose follow. If na important information, e better make human professional translate am. We no go responsible for any misunderstanding or wrong meaning wey fit show because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
